# R&D — Acte d'Engagement : remplissage via GPT-4o Vision

## Pourquoi GPT-4o ?

L'approche classique (mots-clés hardcodés) est fragile car chaque organisme
public marocain utilise sa propre variante du formulaire :
- En-têtes de section différents
- Labels de champs différents (`"patente"` vs `"taxe professionnelle"` vs `"Taxe pro n°"`)
- Champs parfois sur la même ligne, parfois séparés

GPT-4o vision **comprend le document** quel que soit son format.

## Logique en 3 étapes

```
1. PDF → images base64  (pymupdf)
       ↓
2. GPT-4o vision        (1 appel, toutes les pages)
   → "Trouve la section personne morale, retourne les champs + positions Y%"
   → JSON structuré
       ↓
3. pymupdf rawdict      (remplissage précis)
   → Pour chaque champ : trouver la zone de points la plus proche du Y% GPT-4o
   → Effacer + insérer la valeur
```

**Coût estimé :** ~$0.05–0.15 par document selon le nombre de pages.

## Cellule 1 — Config & chargement

In [ ]:
import sys, os, base64, json
from pathlib import Path
import fitz
import openai
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import io

ROOT   = Path("../..")
INPUT  = Path("input")
OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

sys.path.insert(0, str(ROOT))

# ── Clé OpenAI ───────────────────────────────────────────────────────────
# Priorité : variable d'environnement → fichier .env du projet
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
openai.api_key = os.getenv("OPENAI_API_KEY", "")
assert openai.api_key, "❌ OPENAI_API_KEY manquante dans .env"

# ── PDF ──────────────────────────────────────────────────────────────────
pdfs = list(INPUT.glob("*.pdf"))
assert pdfs, "❌ Aucun PDF dans input/"
PDF_PATH = pdfs[0]
print(f"✅ PDF : {PDF_PATH.name}")
print(f"✅ OpenAI key : {openai.api_key[:8]}...")

## Cellule 2 — PDF → images base64

On rasterise chaque page en image PNG et on l'encode en base64 pour l'API GPT-4o.
Un seul appel API avec toutes les pages — GPT-4o voit l'intégralité du document.

In [ ]:
def pdf_to_base64_images(pdf_path: Path, dpi: int = 150) -> list[str]:
    """
    Convertit chaque page du PDF en image PNG encodée base64.

    dpi=150 : bon compromis qualité/taille pour GPT-4o vision.
    Au-delà de 200dpi les images deviennent trop lourdes sans gain de précision.

    Retourne : liste de strings base64 (une par page)
    """
    doc = fitz.open(str(pdf_path))
    images_b64 = []
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    for page in doc:
        pix  = page.get_pixmap(matrix=mat, alpha=False)
        png  = pix.tobytes("png")
        images_b64.append(base64.b64encode(png).decode("utf-8"))
    doc.close()
    print(f"✅ {len(images_b64)} page(s) converties en base64 ({dpi} dpi)")
    return images_b64

pages_b64 = pdf_to_base64_images(PDF_PATH)

## Cellule 3 — Appel GPT-4o : analyse de la structure

GPT-4o reçoit toutes les pages et doit :
1. Trouver la section **personne morale / société privée** (peu importe comment elle s'appelle)
2. Pour chaque champ à remplir dans cette section : retourner le label, la clé métier et la **position Y en %** depuis le haut de la page

Le JSON retourné sera utilisé à l'étape suivante pour guider pymupdf.

In [ ]:
SYSTEM_PROMPT = """
Tu es un expert en analyse de formulaires administratifs marocains (actes d'engagement, déclarations sur l'honneur).
Tu dois analyser le document et retourner UNIQUEMENT du JSON valide, sans aucun texte autour.
""".strip()

USER_PROMPT = """
Ce document est un formulaire de déclaration sur l'honneur / acte d'engagement marocain.

Il contient plusieurs sections (personne physique, personne morale, groupement, établissement public...).
Tu dois identifier UNIQUEMENT la section qui concerne une SOCIÉTÉ PRIVÉE (personne morale, société de droit privé).
Ignore toutes les autres sections (physique, groupement, établissement public, coopérative...).

Pour chaque champ à remplir dans la section société privée, retourne :
- "label"     : le texte exact du libellé dans le document (ex: "Numéro de téléphone")
- "field_key" : une clé parmi cette liste EXACTE :
    signataire_nom, raison_sociale, capital_social, telephone, fax, email,
    adresse_siege, adresse_domicile, cnss, rc_localite, rc_numero,
    taxe_pro, ice, rib, qualite_signataire, autre
- "page"  : numéro de page (0 = première page)
- "y_pct" : position verticale du champ en % depuis le HAUT de la page (0=haut, 100=bas)
- "x_pct" : position horizontale approximative du DÉBUT de la zone à remplir (0=gauche, 100=droite)
- "nth"   : si plusieurs zones de points sur la même ligne, index de celle à remplir (0=première, 1=deuxième...)

ATTENTION AUX FAUX POSITIFS :
- Ne pas inclure les champs des autres sections
- Ne pas confondre les notes de bas de page avec des champs à remplir
- Si téléphone et fax sont sur la même ligne, les retourner séparément avec nth=0 et nth=1

Retourne ce JSON et rien d'autre :
{
  "section_label": "...",
  "fields": [
    {
      "label": "...",
      "field_key": "...",
      "page": 0,
      "y_pct": 42.5,
      "x_pct": 30.0,
      "nth": 0
    }
  ]
}
""".strip()


def analyze_with_gpt4o(pages_b64: list[str]) -> dict:
    """
    Envoie toutes les pages à GPT-4o vision en un seul appel.

    GPT-4o analyse la structure du document et identifie les champs
    de la section personne morale/société avec leurs positions en %.

    Retourne : dict parsé depuis le JSON GPT-4o
    """
    # Construction du message : system + toutes les images + prompt texte
    image_contents = [
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/png;base64,{b64}",
                "detail": "high",   # haute résolution pour lire les petits textes
            }
        }
        for b64 in pages_b64
    ]
    image_contents.append({"type": "text", "text": USER_PROMPT})

    client = openai.OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": image_contents},
        ],
        response_format={"type": "json_object"},
        max_tokens=2000,
        temperature=0,   # déterministe — on veut toujours le même résultat
    )

    raw = response.choices[0].message.content
    result = json.loads(raw)
    tokens = response.usage.total_tokens
    print(f"✅ GPT-4o répondu — {tokens} tokens utilisés")
    print(f"   Section détectée : {result.get('section_label', '?')}")
    print(f"   Champs trouvés   : {len(result.get('fields', []))}")
    return result


gpt_result = analyze_with_gpt4o(pages_b64)
print("\n── Réponse GPT-4o brute ──────────────────────────────")
print(json.dumps(gpt_result, ensure_ascii=False, indent=2))

## Cellule 4 — Visualisation des positions GPT-4o

On affiche les positions retournées par GPT-4o sur les pages du document.
Les croix rouges = position Y% GPT-4o → doit tomber sur la bonne ligne.

In [ ]:
def visualize_gpt_positions(pages_b64: list[str], fields: list[dict]):
    """
    Affiche les pages avec les positions retournées par GPT-4o.
    Croix rouge = position (x_pct, y_pct) indiquée par GPT-4o.
    Label = field_key pour identifier visuellement chaque champ.
    """
    # Grouper les champs par page
    by_page: dict[int, list] = {}
    for f in fields:
        by_page.setdefault(f["page"], []).append(f)

    for p_idx, b64 in enumerate(pages_b64):
        png_bytes = base64.b64decode(b64)
        img = Image.open(io.BytesIO(png_bytes))
        w, h = img.size

        fig, ax = plt.subplots(figsize=(8, 11))
        ax.imshow(img)
        ax.axis("off")

        for f in by_page.get(p_idx, []):
            px = f["x_pct"] / 100 * w
            py = f["y_pct"] / 100 * h
            # Croix rouge à la position GPT-4o
            ax.plot(px, py, "r+", markersize=14, markeredgewidth=2)
            # Label field_key en rouge à côté
            ax.text(px + 5, py, f["field_key"], fontsize=7, color="red",
                    va="center", bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))

        ax.set_title(f"Page {p_idx + 1} — positions GPT-4o (croix rouges)", fontsize=10)
        plt.tight_layout()
        plt.show()


visualize_gpt_positions(pages_b64, gpt_result.get("fields", []))

## Cellule 5 — Remplissage guidé par GPT-4o

Pour chaque champ retourné par GPT-4o :
1. Convertir `y_pct` → position Y absolue sur la page pymupdf
2. Chercher dans rawdict la ligne dont le Y est le plus proche de cette position
3. Dans cette ligne, prendre la `nth` zone de points
4. Effacer + insérer la valeur

C'est pymupdf qui fait le remplissage réel — GPT-4o ne fait qu'indiquer où chercher.

In [ ]:
_DOT_CHARS = set('._…\u2026\u00b7·\u2025\u22ef\u0085')

# ── Données à insérer — modifier selon votre dossier ─────────────────────
DATA = {
    "signataire_nom"    : "Mohammed Benali",
    "qualite_signataire": "Directeur Général",
    "raison_sociale"    : "ABI Consulting SARL AU",
    "capital_social"    : "100.000 MAD",
    "telephone"         : "0522 123 456",
    "fax"               : "0522 789 012",
    "email"             : "contact@abi-consulting.ma",
    "adresse_siege"     : "Imm 30, Appt 08, Rue Moulay Ahmed Loukili, Rabat",
    "adresse_domicile"  : "12 Rue Hassan II, Casablanca",
    "cnss"              : "1234567",
    "rc_localite"       : "Casablanca",
    "rc_numero"         : "RC 145853",
    "taxe_pro"          : "TP 56789012",
    "ice"               : "002579010000023",
    "rib"               : "007780000001234567890123",
}


def find_dot_zones_in_spans(spans: list) -> list[tuple]:
    """
    Détecte les zones de points consécutifs dans une ligne (rawdict).
    Étend l'effacement aux hints entre parenthèses qui suivent les points.
    Retourne : liste de (x0_insert, y0, x1_erase, y1, fontsize)
    """
    zones = []
    for span in spans:
        chars = span.get("chars", [])
        i = 0
        while i < len(chars):
            if chars[i]["c"] not in _DOT_CHARS:
                i += 1
                continue
            dot_start = i
            while i < len(chars) and chars[i]["c"] in _DOT_CHARS:
                i += 1
            dot_end = i - 1
            if dot_end - dot_start + 1 < 3:
                continue
            x0 = chars[dot_start]["bbox"][0]
            x1 = chars[dot_end]["bbox"][2]
            y0 = min(c["bbox"][1] for c in chars[dot_start:dot_end + 1])
            y1 = max(c["bbox"][3] for c in chars[dot_start:dot_end + 1])
            # Étendre à "(hint)" qui suit immédiatement
            erase_x1 = x1
            if i < len(chars) and chars[i]["c"] == '(':
                j = i
                while j < len(chars) and chars[j]["c"] != ')':
                    j += 1
                if j < len(chars):
                    erase_x1 = chars[j]["bbox"][2]
                    i = j + 1
            zones.append((x0, y0, erase_x1, y1, span.get("size", 9.5)))
    return zones


def fill_field_at_y(page: fitz.Page, y_target_abs: float, nth: int, value: str,
                    y_tolerance: float = 20.0) -> bool:
    """
    Remplit la nth zone de points sur la ligne la plus proche de y_target_abs.

    y_target_abs : Y absolu en points PDF (converti depuis y_pct GPT-4o)
    y_tolerance  : marge en points pour accepter une ligne comme "proche"
                   (20pt ≈ 7mm — suffisant pour absorber l'imprécision GPT-4o)

    Retourne True si un remplissage a été effectué, False sinon.
    """
    best_line = None
    best_dist = float("inf")

    for block in page.get_text("rawdict").get("blocks", []):
        for line in block.get("lines", []):
            spans = line.get("spans", [])
            if not spans:
                continue
            # Y du milieu de la ligne
            line_y = (line["bbox"][1] + line["bbox"][3]) / 2
            dist = abs(line_y - y_target_abs)
            zones = find_dot_zones_in_spans(spans)
            if zones and dist < best_dist and dist < y_tolerance:
                best_dist = dist
                best_line = (spans, zones)

    if best_line is None:
        return False

    spans, zones = best_line
    if nth >= len(zones):
        # nth hors limite → prendre la dernière zone disponible
        nth = len(zones) - 1

    x0, y0, x1_erase, y1, fs = zones[nth]
    page.draw_rect(
        fitz.Rect(x0 - 1, y0 - 1, x1_erase + 1, y1 + 1),
        color=(1, 1, 1), fill=(1, 1, 1),
    )
    page.insert_text(
        fitz.Point(x0, y1 - 1),
        value,
        fontsize=max(fs, 7.0),
        color=(0, 0, 0),
    )
    return True


def fill_from_gpt_result(pdf_path: Path, gpt_fields: list[dict], data: dict) -> bytes:
    """
    Remplit le PDF en utilisant les positions retournées par GPT-4o.

    Pour chaque champ GPT-4o :
    - Récupère la valeur correspondante dans data[field_key]
    - Convertit y_pct → Y absolu sur la page pymupdf
    - Appelle fill_field_at_y pour remplir la zone de points la plus proche

    Retourne : bytes du PDF rempli
    """
    doc = fitz.open(str(pdf_path))
    filled, skipped = 0, 0

    for f in gpt_fields:
        field_key = f.get("field_key", "")
        value     = data.get(field_key, "")
        page_idx  = f.get("page", 0)
        y_pct     = f.get("y_pct", 50)
        nth       = f.get("nth", 0)

        if not value:
            print(f"  ⚠️  {field_key} — pas de valeur dans DATA, ignoré")
            skipped += 1
            continue

        if page_idx >= len(doc):
            print(f"  ⚠️  {field_key} — page {page_idx} inexistante, ignoré")
            skipped += 1
            continue

        page       = doc[page_idx]
        page_h     = page.rect.height
        y_abs      = y_pct / 100 * page_h   # conversion % → points PDF

        ok = fill_field_at_y(page, y_abs, nth, value)
        if ok:
            print(f"  ✅ {field_key:25s} → '{value[:40]}'")
            filled += 1
        else:
            print(f"  ❌ {field_key:25s} — aucune zone de points trouvée à Y={y_abs:.0f}pt")
            skipped += 1

    print(f"\nRésultat : {filled} champs remplis, {skipped} ignorés")
    out = io.BytesIO()
    doc.save(out)
    doc.close()
    return out.getvalue()


result_bytes = fill_from_gpt_result(PDF_PATH, gpt_result.get("fields", []), DATA)

## Cellule 6 — Avant / Après + Sauvegarde

In [ ]:
def page_to_img(page, dpi=120):
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    pix = page.get_pixmap(matrix=mat, alpha=False)
    return Image.frombytes("RGB", [pix.width, pix.height], pix.samples)


doc_orig   = fitz.open(str(PDF_PATH))
doc_result = fitz.open(stream=result_bytes, filetype="pdf")

for p_idx in range(doc_orig.page_count):
    fig, axes = plt.subplots(1, 2, figsize=(16, 11))
    axes[0].imshow(page_to_img(doc_orig[p_idx]))
    axes[0].set_title(f"AVANT — Page {p_idx + 1}", fontsize=10)
    axes[0].axis("off")
    axes[1].imshow(page_to_img(doc_result[p_idx]))
    axes[1].set_title(f"APRÈS (GPT-4o) — Page {p_idx + 1}", fontsize=10)
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

# Sauvegarde
out_path = OUTPUT / "acte_engagement_gpt4o.pdf"
out_path.write_bytes(result_bytes)
print(f"✅ Sauvegardé → {out_path}")